In [ ]:
%env JAX_PLATFORMS=cpu
%env JAX_ENABLE_X64=True

In [ ]:
import sys
from pathlib import Path

from IPython.display import SVG, display

cwd = Path.cwd()
root = next(
    (path for path in (cwd, *cwd.parents) if (path / "experiments/capacity_bound/capacity_bound.py").is_file()),
    None,
)
if root is None:
    raise RuntimeError("Open this notebook from the repository or its experiment folder.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from experiments.capacity_bound import capacity_bound as bench

# Run experiments

Generate the paper's capacity and recurrent-current figures, then run the simulator and weight-bound checks. The [README](README.md) lists all six figures and the full gradient validation.

## Configuration

The capacity figure compares simulated output counts with the exact gap count. The recurrent-current figure shows how delayed re-entry changes the output train. Set `FIGURE_NAMES = tuple(bench.FIGURES)` for all six figures. The pseudospike calculation reuses its cache unless `RECOMPUTE_PSEUDOSPIKE` is enabled.

In [ ]:
FIGURE_NAMES = ("capacity", "r1_train")
OUTPUT_DIR = bench.OUTPUT_DIR
RECOMPUTE_PSEUDOSPIKE = False
RUN_FULL_VALIDATION = False

## Generate figures

In [ ]:
paths = bench.generate_figures(
    FIGURE_NAMES, OUTPUT_DIR, recompute=RECOMPUTE_PSEUDOSPIKE,
)

# Plot results

## Paper figures

In [ ]:
for path in paths:
    display(SVG(filename=str(path)))

# Validate results

The checks compare the complex-step event Jacobian with the closed form, compare RTRL traces with finite differences, and evaluate the weight bounds. Set `RUN_FULL_VALIDATION = True` to also run the paper's gradient checks against JAX autodiff, high-precision derivatives, and independent simulations. Both figures and checks add sections to `RUN_RESULTS.md`.

In [ ]:
gradient_records = bench.run_checks(OUTPUT_DIR, full=RUN_FULL_VALIDATION)
report = Path(OUTPUT_DIR) / "RUN_RESULTS.md"
print(f"Report saved to {report.relative_to(root) if report.is_relative_to(root) else report}")